# Mimir, hands on

[Mimir](https://github.com/AshNicolus/mimir) gives an agent memory of its own experience: what was tried, on which task, and how it turned out. This notebook walks the whole loop, from recording experiences to letting an agent explore new strategies.

```
pip install mimir-learn
```

Everything below runs top to bottom with no extras and no API keys.

In [1]:
from mimir import Mimir

# An in-memory store for the demo. Use a file path to persist across runs.
memory = Mimir(":memory:")

## 1. Record what happened

Every experience is a task, the action taken, and how it went. Failures are recorded with a reason, which is what a future run reads to avoid repeating them.

In [2]:
memory.record(
    task="fix login latency under load",
    action="add a redis cache in front of session lookups",
    outcome="success",
    score=0.9,
)
memory.record_failure(
    task="throttle abusive clients",
    action="add a fixed-window rate limiter",
    reason="WebSocket traffic wasn't handled; the limiter only saw HTTP",
)
print(memory.count(), "experiences stored")

2 experiences stored


## 2. Recall relevant experience

`recall` finds the most relevant past experiences for a query. Filter by outcome to study just the wins or just the mistakes.

In [3]:
for exp in memory.recall("login is slow", k=3):
    print(f"{exp.outcome.value:8s} | {exp.action}")

for exp in memory.recall("rate limiting", outcome="failure"):
    print("known failure:", exp.context["failure_reason"])

success  | add a redis cache in front of session lookups
known failure: WebSocket traffic wasn't handled; the limiter only saw HTTP


## 3. Ask what to do

`recommend` aggregates the track record per action and returns the best-supported one. Confidence is the lower bound of a Beta posterior on the action's success rate, so it reads as a conservative success rate: one win is weak evidence, a clean streak earns real confidence.

In [4]:
for _ in range(7):
    memory.record("api latency high under load", "add a redis cache", outcome="success")
memory.record("api latency high under load", "raise the thread count", outcome="failure")

rec = memory.recommend("api latency high under load")
print(rec)
print("supported by", len(rec.supporting_ids), "experiences")

Recommended strategy: 'add a redis cache'
  confidence: 0.77
  based on 7 experiences (7 success / 0 failure)
supported by 7 experiences


A simple gate makes this actionable: reuse the recommendation when confidence is high, explore when it is low.

In [5]:
rec = memory.recommend("api latency high under load")
if rec and rec.confidence > 0.6:
    print("reuse:", rec.recommended_action)
else:
    print("explore something new")

reuse: add a redis cache


## 4. Keep memory fresh

Knowledge goes stale two ways, and Mimir handles both. Supersede an experience explicitly, or set a half-life so old evidence loses weight on its own.

In [6]:
old = memory.record("auth is slow", "add a write-through cache", outcome="failure")
new = memory.record("auth is slow", "add a read cache", supersedes=old.id)
print("superseded rows are hidden from recall:", [e.action for e in memory.recall("auth slow")])

superseded rows are hidden from recall: ['add a read cache']


In [7]:
from datetime import timedelta

from mimir import Experience
from mimir.models import utcnow

fresh = Mimir(":memory:", half_life_days=30)

def record_aged(task, action, days_ago, n):
    for _ in range(n):
        fresh.write(Experience(task=task, action=action, created_at=utcnow() - timedelta(days=days_ago)))

record_aged("scale the service", "manual scaling", days_ago=300, n=4)  # proven long ago
record_aged("scale the service", "autoscaling", days_ago=5, n=2)      # winning lately

print("with decay:", fresh.recommend("scale the service").recommended_action)
fresh.half_life_days = None
print("without   :", fresh.recommend("scale the service").recommended_action)
fresh.close()

with decay: autoscaling
without   : manual scaling


## 5. Explore instead of always exploiting

The default mode always returns the best-supported action, so an almost-as-good newcomer never gets a chance. `explore=True` draws the winner by Thompson sampling: each action's rank is a draw from its posterior, so less-proven actions win a share of calls proportional to how likely they are to actually be better. Actions that have only ever failed are still never recommended.

In [8]:
import random
from collections import Counter

lab = Mimir(":memory:")
for _ in range(20):
    lab.record("api latency", "proven fix", outcome="success")     # 20/0 veteran
lab.record("api latency", "newcomer fix", outcome="success")       # 1/0 newcomer
lab.record("api latency", "broken fix", outcome="failure")         # 0/1, never recommended

picks = Counter(
    lab.recommend("api latency", explore=True, rng=random.Random(i)).recommended_action
    for i in range(200)
)
for action, n in picks.most_common():
    print(f"{action:14s} picked {n}/200 times")

proven fix     picked 171/200 times
newcomer fix   picked 29/200 times


The veteran still wins most calls, the newcomer gets a real share, and the pure failure never appears. Close the loop by recording each explored outcome, and the newcomer either builds a track record or gets ruled out.

## 6. Distill a conversation into an experience

Agents usually end a task holding a transcript, not a clean task/action pair. A distiller bridges the two. Here a toy heuristic stands in for an LLM; ground truth for the outcome always comes from the caller.

In [9]:
from mimir import CallableDistiller, Draft

def toy_distiller(messages):
    task = messages[0]["content"]
    action = next(m["content"] for m in reversed(messages) if m["role"] == "assistant")
    return Draft(task=task, action=action)

transcript = [
    {"role": "user", "content": "checkout page times out during sales"},
    {"role": "assistant", "content": "moved cart reads to a replica and cached product data"},
    {"role": "user", "content": "latency is back under 200ms, thanks"},
]

exp = memory.record_conversation(
    transcript,
    distiller=CallableDistiller(toy_distiller, name="toy"),
    outcome="success",   # ground truth from your evaluation, not the LLM
)
print("stored:", exp.action)
print("provenance:", exp.context["source"], "/", exp.context["distiller"])

stored: moved cart reads to a replica and cached product data
provenance: transcript / toy


## Wrap-up

That is the whole loop: record outcomes, recall evidence, act on recommendations, explore when evidence is thin, and keep the store fresh with superseding and decay.

- [README](../README.md) for the design and why experience memory matters
- [PLAYBOOK](../PLAYBOOK.md) for the full basic-to-advanced guide, including LLM and LangChain/LangGraph integrations

In [10]:
memory.close()